# SARSA (State–Action–Reward–State–Action) and temporal-difference learning

_Artificial Intelligence — more_

**Don't wait for the episode to end. Update each step from the very next step's guess.**

Monte Carlo waits for the whole episode, then averages. Temporal-difference (TD) learning updates after every single step.

     It nudges a value toward the reward just seen plus the discounted value of the next state. This is called bootstrapping: using one estimate to improve another.

---

This notebook is a practice scaffold for the **SARSA (State–Action–Reward–State–Action) and temporal-difference learning** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

### Step 1 — Set up the corridor and its values

We have a 5-cell corridor where reaching the last cell is worth `+1`. The array `V` holds our current value estimate for each cell; we pin the goal cell's value at `1.0` and start everything else at zero.

`alpha` is the learning rate (how far each update moves the estimate) and `gamma` is the discount that makes value shrink with distance from the goal.

In [ ]:
N = 5            # number of cells in the corridor
alpha = 0.5      # learning rate for each TD update
gamma = 0.9      # discount factor

V = np.zeros(N)
V[N - 1] = 1.0   # goal cell value fixed at +1

### Step 2 — Sweep the TD(0) update

Temporal-difference learning updates a cell from the *very next* cell's current estimate instead of waiting for the episode to finish. The TD(0) rule is `V(s) <- V(s) + alpha[ r + gamma·V(s') - V(s) ]`; here every step's reward `r` is 0, so all the value comes from the discounted goal.

We sweep right-to-left so the goal's value propagates backward one cell per sweep, and repeat for 20 sweeps until the estimates settle.

In [ ]:
# TD(0) value update: V(s) <- V(s) + alpha[ r + gamma V(s') - V(s) ], reward r = 0.
for sweep in range(20):
    for s in range(N - 2, -1, -1):       # sweep right-to-left so value flows back
        reward = 0
        target = reward + gamma * V[s + 1]   # bootstrap off the next cell's value
        td_error = target - V[s]
        V[s] += alpha * td_error

### Step 3 — Compare against the true values

With a deterministic walk and no step reward, the exact value of a cell is just `gamma` raised to its distance from the goal. We compute that closed-form answer and check our learned `V` against it.

After 20 sweeps the maximum error should be tiny — TD(0) has bootstrapped its way to the correct discounted values.

In [ ]:
print("learned V:", [round(v, 3) for v in V])

# True value of each cell = gamma^(distance to goal).
true = np.array([gamma ** (N - 1 - s) for s in range(N)])
print("true   V:", [round(v, 3) for v in true])

max_error = float(np.max(np.abs(V - true)))
print("max error:", round(max_error, 4))

## Visualize the data & results

_Warehouse delivery robot: how does value flow back from the loading dock to the starting shelf over TD(0) sweeps?_

### Step 1 — Track the start shelf's value over sweeps

Same corridor, framed as a warehouse robot: shelf 4 is the loading dock (reward `+1`) and we watch value flow back to shelf 0.

We rerun the TD(0) sweeps, but this time record `V[0]` after each sweep into `track` so we can plot how the start shelf's estimate climbs toward its true value.

In [ ]:
# Warehouse robot on a 5-shelf corridor; shelf 4 is the loading dock (reward +1).
N, alpha, gamma = 5, 0.5, 0.9
V = np.zeros(N)
V[N - 1] = 1.0            # dock value pinned at +1

track = []
for sweep in range(1, 21):
    for s in range(N - 2, -1, -1):     # right-to-left so value flows back toward the start
        target = gamma * V[s + 1]      # discounted value of the next shelf (reward 0)
        V[s] += alpha * (target - V[s])
    track.append(V[0])

### Step 2 — Plot the climb toward the discounted reward

Because value spreads back only one shelf per sweep, the start shelf stays near zero for the first few sweeps and then rises, leveling off at its true value `gamma^4 = 0.656`.

We sample `track` at a handful of sweep counts and plot them against the converged value to make that backward flow visible.

In [ ]:
xs = [1, 2, 3, 5, 10, 20]
ys = [track[x - 1] for x in xs]

true_v0 = gamma ** (N - 1)             # 0.656

fig, ax = plt.subplots()
ax.plot(xs, ys, "-o", color="#4ea1ff", label="V(shelf 0)")
ax.axhline(true_v0, color="#7ee787", label="true value gamma^4 = 0.656")
ax.set_xlabel("TD(0) sweep")
ax.set_ylabel("value of start shelf")
ax.set_title("V(start shelf) climbing toward the discounted dock reward over TD sweeps")
ax.legend()
plt.show()